# B2-024 — Practice p12

*50 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

**Set:** B · **Type:** constrained-coding · **Difficulty:** advanced · **Time budget:** 50 minutes  
**Concepts:** open-ended-experiment-design

## Task

Implement an ablation runner that enforces an equal step budget and reports mean ± sample standard deviation.

**Contract.** `run_ablation(configs, train_fn, seeds, steps_per_run)`:

* `configs` is a `dict` from config name to a config object; `seeds` is a sequence of at least two ints (raise `ValueError` otherwise);
* for each config (in dict order) and each seed (in order), create a fresh `capstone_data.StepBudget(steps_per_run)` and call `score = float(train_fn(config, seed, budget))`; `train_fn` must wrap every optimizer it creates with `budget.wrap(...)`;
* after each call, if `budget.used != steps_per_run`, raise `RuntimeError` (every run must spend exactly the same budget);
* return a `dict` from config name to `{"scores": tuple of floats in seed order, "mean": float, "std": float, "steps": int}`, where `"std"` is the **sample** standard deviation (divisor $r-1$) and `"steps"` is the total steps for that config.

**Toy `train_fn`.** Write `toy_train_fn(config, seed, budget)`: `w = torch.tensor(float(seed), requires_grad=True)`, `optimizer = budget.wrap(torch.optim.SGD([w], lr=config["lr"]))`; while `budget.remaining > 0`: zero the gradient, compute `loss = (w - 3.0) ** 2`, call `backward()` and `step()`; return `float((w.detach() - 3.0) ** 2)`.

**Probes.** With `TOY_CONFIGS`, `seeds = (0, 1, 2)`, and `steps_per_run = 5`, the table must equal (`math.isclose`, `rel_tol = 1e-6`, `abs_tol = 1e-9`):

| config | scores | mean | std |
|---|---|---:|---:|
| `lr=0.1` | 0.9663678407669067, 0.429496705532074, 0.10737410187721252 | 0.5010795493920644 | 0.4339477370769859 |
| `lr=0.2` | 0.05441953241825104, 0.024186458438634872, 0.006046614609658718 | 0.028217535155514877 | 0.024437102507298938 |
| `lr=0.05` | 3.138105869293213, 1.3947136402130127, 0.3486785590648651 | 1.6271660228570302 | 1.4091670456016205 |

and `"steps" == 15` for every config.
Also: a `train_fn` that stops one step early raises `RuntimeError`; `seeds = (0,)` raises `ValueError`; and a wrong variant using the population standard deviation (divisor $r$) gives `0.3543168436246828` for `lr=0.1`, which must not match.

**A real ablation.** Finally, use `run_ablation` on the inverse task with two configs of your choice that differ in exactly one factor (an MLP no wider than 32, full-batch Adam, `steps_per_run = 100`, seeds `(0, 1)`, scored by mean **validation** position error), print the table, and state in two sentences whether the difference is demonstrated.

**Required answer-check assertions.** The toy table (scores, means, stds, steps), the two exceptions, the population-std mismatch, and that the real ablation's table has two configs with `"steps" == 200` each.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)

import math

TOY_CONFIGS = {"lr=0.1": {"lr": 0.1}, "lr=0.2": {"lr": 0.2}, "lr=0.05": {"lr": 0.05}}

In [ ]:
# Write the required implementation, probes, certificates, and answer-check assertions here.

## Your response

Show every requested derivation, implementation, probe, certificate, or audit. Use only the permitted data, keep the training, validation, and locked-test roles separate, and end coding work with an answer-check section of executable assertions.